# Stdio and the stderr rule

**Scenario:** a container terminal runs a berth planner. Its MCP server answers one question: how long
a ship waits for a quay. It worked for a month. Somebody added a line of tracing, and the planner now
freezes on every lookup, with no error anywhere.

A transport is how two programs carry messages to each other. Over stdio the host and the server share
one pipe, so think of **two people writing on the same sheet of paper**. While one writes, the page
reads fine. The moment the other adds a word, the sentence belongs to nobody.

## Mechanics

Stdio means talking over the input and output streams of the process. The host starts the server as a
child, and the pipes are the whole connection.

| Stream | Who writes it | What is allowed |
|---|---|---|
| `stdin` | The host | One JSON-RPC request per line |
| `stdout` | The server | The protocol, and nothing else. One JSON object per line |
| `stderr` | The server | Anything. Logs, warnings, tracebacks. The host may show it or drop it |

| Rule | Detail |
|---|---|
| Framing | Newline delimited. A message may not contain a raw newline inside it |
| Handshake | `initialize`, then the `notifications/initialized` message, then requests |
| Shutdown | The host closes `stdin`. The server exits, or it gets a signal |

There is a second transport, and the difference that matters is not speed.

| Transport | Where the server runs | Who starts it | How it knows who is calling |
|---|---|---|---|
| stdio | The same machine as the host | The host, as a child process | It does not. The child inherits the account that started it |
| Streamable HTTP | Anywhere | Nobody. It is already running | An `Authorization` header, normally an OAuth access token |

Read the last column again. Stdio has no request header, so it has nowhere to put a token. Whatever
the host account can reach, the server can reach. That is the whole access model.

## The picture

![stdout is the protocol lane and stderr is the free lane](images/stdio-framing.svg)

Two lanes leave the server. Only one is parsed, and only one is safe to write on.

## The cost

```
wasted = requests after the first bad reply x the host request timeout
```

A corrupted reply is not an error. It does not come back at all, so the host pays its full timeout on
every call until somebody restarts the server.

## The failure

One function decides where a trace goes, chosen by an environment variable. That is the only
difference between the version that works and the version that hangs.

In [1]:
import pathlib

SERVER = pathlib.Path("servers/berths.py")

body = SERVER.read_text()
print(body[body.index("def trace"):body.index("@srv.tool")].rstrip())

def trace(message: str) -> None:
    """Where a diagnostic goes. This is the whole lesson."""
    if MODE == "stdout-line":
        print(message, flush=True)
    elif MODE == "stdout-partial":
        print(message, end="", flush=True)
    else:
        print(message, file=sys.stderr, flush=True)


To see what each choice does we need a real host, so we use the SDK client. It is asynchronous and a
notebook already owns an event loop, so the work runs in a worker thread with its own loop and a time
limit.

In [2]:
import asyncio
import threading


def run_async(coro, limit=25):
    """Run one coroutine on its own loop, in a thread that cannot outlive us."""
    box = {}

    def worker():
        try:
            box["value"] = asyncio.run(asyncio.wait_for(coro, limit))
        except BaseException as exc:
            box["error"] = exc

    thread = threading.Thread(target=worker, daemon=True)
    thread.start()
    thread.join(limit + 10)
    error = box.get("error")
    while isinstance(error, BaseExceptionGroup) and len(error.exceptions) == 1:
        error = error.exceptions[0]        # a group of one is noise, not structure
    if error:
        raise error
    return box["value"]

Now the host. It starts the server, does the handshake, asks one question, and keeps whatever the
server wrote to stderr.

In [3]:
import sys
import tempfile
import time
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client


async def probe(mode, seconds=6):
    """One full exchange with the berth server, run under a short deadline."""
    params = StdioServerParameters(command=sys.executable, args=[str(SERVER)],
                                   env={"BERTH_LOG": mode, "PYTHONWARNINGS": "ignore"})
    log = tempfile.TemporaryFile("w+")
    async with stdio_client(params, errlog=log) as (read, write):
        async with ClientSession(read, write) as session:
            await asyncio.wait_for(session.initialize(), seconds)
            reply = await asyncio.wait_for(
                session.call_tool("berth_wait_hours", {"terminal": "NLRTM"}), seconds)
    log.seek(0)
    return reply.structuredContent, log.read().strip().splitlines()

Run it twice. First with the trace on stderr, the version that shipped. Then on stdout with no
trailing newline, the version somebody added.

In [4]:
print("stderr        ->", run_async(probe("stderr")))

started = time.monotonic()
try:
    print("stdout-partial->", run_async(probe("stdout-partial")))
finally:
    print(f"the host waited {time.monotonic() - started:.1f} seconds and got nothing back")

stderr        -> ({'result': 6.5}, ['[berths] wait lookup for NLRTM'])


Failed to parse JSONRPC message from server
Traceback (most recent call last):
  File "/Users/param/learn/learnwithparam/lwp-repos/ai-engineering-vaults/.venv/lib/python3.11/site-packages/mcp/client/stdio/__init__.py", line 155, in stdout_reader
    message = types.JSONRPCMessage.model_validate_json(line)
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/param/learn/learnwithparam/lwp-repos/ai-engineering-vaults/.venv/lib/python3.11/site-packages/pydantic/main.py", line 766, in model_validate_json
    return cls.__pydantic_validator__.validate_json(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
pydantic_core._pydantic_core.ValidationError: 1 validation error for JSONRPCMessage
  Invalid JSON: expected value at line 1 column 2 [type=json_invalid, input_value='[berths] wait lookup for...:6.5},"isError":false}}', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/json_invalid


the host waited 7.7 seconds and got nothing back


TimeoutError: 

## The diagnosis

The server did not crash. It answered, and the answer was destroyed on the way out by the process that
produced it.

**Framing is the mechanic.** Stdout is newline delimited, so the host reads to the next newline and
parses what it has. A trace ending in a newline becomes its own line, thrown away with a parse error.
A trace without one has no boundary, so the reply is glued to it and the whole line is unparseable.

**The reply is not late, it is gone.** Nothing retransmits. The host waits on an `id` that will never
arrive, so its only way out is its own clock. The symptom is a freeze.

Here is the same server driven by hand, so the bytes are visible.

In [5]:
import json
import os
import subprocess

HELLO = {"jsonrpc": "2.0", "id": 1, "method": "initialize",
         "params": {"protocolVersion": "2025-06-18", "capabilities": {},
                    "clientInfo": {"name": "by-hand", "version": "0"}}}
ASK = {"jsonrpc": "2.0", "id": 2, "method": "tools/call",
       "params": {"name": "berth_wait_hours", "arguments": {"terminal": "NLRTM"}}}
SCRIPT = "".join(json.dumps(m) + "\n" for m in
                 (HELLO, {"jsonrpc": "2.0", "method": "notifications/initialized"},
                  ASK, dict(ASK, id=3)))


def stdout_of(mode):
    """Every line the server wrote to stdout, exactly as the host would read it."""
    done = subprocess.run([sys.executable, str(SERVER)], input=SCRIPT, text=True,
                          capture_output=True, timeout=20,
                          env={**os.environ, "BERTH_LOG": mode, "PYTHONWARNINGS": "ignore"})
    return done.stdout.splitlines()


for mode in ("stdout-line", "stdout-partial"):
    print(f"--- {mode}")
    for line in stdout_of(mode):
        print("   ", line[:96])

--- stdout-line


    {"jsonrpc":"2.0","id":1,"result":{"protocolVersion":"2025-06-18","capabilities":{"experimental":
    [berths] wait lookup for NLRTM
    [berths] wait lookup for NLRTM
    {"jsonrpc":"2.0","id":2,"result":{"content":[{"type":"text","text":"6.5"}],"structuredContent":{
--- stdout-partial


    {"jsonrpc":"2.0","id":1,"result":{"protocolVersion":"2025-06-18","capabilities":{"experimental":
    [berths] wait lookup for NLRTM[berths] wait lookup for NLRTM{"jsonrpc":"2.0","id":2,"result":{"c


## The fix

Look at the last line above. The reply is still there, wearing a trace as a prefix, which is why no
host will find it.

The fix is one keyword. Every diagnostic takes `file=sys.stderr`, and that holds for anything else in
the process that prints, including a library you did not write.

In [6]:
import logging

logging.getLogger("mcp.client.stdio").setLevel(logging.CRITICAL)


def outcome(mode):
    """What a host gets from one call, and how long it waited for it."""
    started = time.monotonic()
    try:
        value, log = run_async(probe(mode))
        verdict = f"{value}, trace lines on stderr: {len(log)}"
    except BaseException as exc:
        verdict = f"{type(exc).__name__}, nothing returned"
    return mode, verdict, time.monotonic() - started

Three runs of the same server, one keyword apart. The client logs a parse error for every line it
discards, and we have seen those, so the table quiets that logger.

In [7]:
for mode in ("stdout-partial", "stdout-line", "stderr"):
    name, verdict, seconds = outcome(mode)
    print(f"{name:16} {seconds:5.1f}s  {verdict}")

stdout-partial     7.6s  TimeoutError, nothing returned


stdout-line        2.1s  {'result': 6.5}, trace lines on stderr: 0


stderr             2.1s  {'result': 6.5}, trace lines on stderr: 1


## The gate

The middle row is worth a second look. That trace went to stdout, so nothing reached stderr and the
diagnostic was swallowed by the protocol lane. The call worked, and the log is gone.

A rule that lives in a review comment gets broken by the next person in a hurry. This one is
checkable: a good server writes nothing on stdout that is not JSON.

In [8]:
def test_stdout_carries_only_the_protocol():
    lines = stdout_of("stderr")
    assert lines, "the server wrote nothing at all"
    for line in lines:
        json.loads(line)


test_stdout_carries_only_the_protocol()
print(f"gate holds: {len(stdout_of('stderr'))} lines on stdout, all of them JSON")

gate holds: 2 lines on stdout, all of them JSON


Point `trace` at stdout and this test fails on the first stray line. Run it against every server you
ship: a terminal merges the two lanes, so on a laptop the trace looks fine.

### Enterprise exploration

- A local server runs as your own account with no token anywhere. Who reviews one before it lands on an
  engineer's laptop, and what is the compliance position if nobody does?
- The host waits out its timeout on every call after the stream breaks. What does that cost during an
  incident, and what would you page on to catch it in seconds rather than hours?
- A dependency prints a deprecation warning to stdout on import. How do you find that before a customer
  does, and what in your build stops it shipping?

### Key takeaways

- Over stdio, stdout is the protocol. Everything else in the process uses stderr.
- A trace ending in a newline is dropped with a parse error. One without a newline eats the next reply.
- The symptom is a freeze, not an error, because a lost reply is never retransmitted.
- Stdio has no header, so it has no token. The server runs as you.